# Environmental Data Analysis: Water Quality Assessment

This notebook cleans water quality data, calculates monthly averages, and visualizes key environmental parameters using Python, Pandas, and Matplotlib.

## Step 1: Import Required Libraries

Import pandas for data manipulation, matplotlib and seaborn for visualization, and datetime for time-based operations.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Set visual style for professional-looking charts
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("Set2")

print("Libraries imported successfully!")

## Step 2: Load and Inspect the Data

Read the CSV file and perform initial exploration to understand the dataset structure, data types, and potential issues.

In [ ]:
# Read the CSV file containing environmental monitoring data
df = pd.read_csv('Water_Quality_Dataset.csv')

# Display initial dataset information
print("Dataset shape:", df.shape)
print("\nColumn names:", df.columns.tolist())
print("\nFirst 5 rows:")
display(df.head())

print("\nData types:")
print(df.dtypes)

## Step 3: Initial Data Exploration

Check for missing values, duplicates, and get summary statistics to understand the data distribution.

In [ ]:
# Check for missing values across all columns
print("Missing values per column:")
print(df.isnull().sum())

# Check for duplicate rows
print(f"\nDuplicate rows: {df.duplicated().sum()}")

# View summary statistics for numeric columns
print("\nSummary statistics:")
display(df.describe())

## Step 4: Data Cleaning

Clean the data by:
1. Converting timestamp to datetime format
2. Converting numeric columns and handling missing values with median imputation
3. Removing duplicate rows

In [ ]:
# Convert timestamp column to datetime format for time-based analysis
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Define numeric columns that need cleaning
numeric_columns = ['Turbidity', 'temperature', 'DO', 'BOD', 'Lead', 'Mercury', 'Arsenic']

# Convert columns to numeric, coercing errors to NaN, then fill missing with median
for col in numeric_columns:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f"{col}: filled {df[col].isnull().sum()} missing values with median {median_val:.2f}")

# Handle Pollution_Level - convert to numeric if needed
df['Pollution_Level'] = pd.to_numeric(df['Pollution_Level'], errors='coerce')
df['Pollution_Level'] = df['Pollution_Level'].fillna(df['Pollution_Level'].median())

# Remove duplicate rows to ensure data integrity
print(f"\nDuplicate rows before cleaning: {df.duplicated().sum()}")
df = df.drop_duplicates()
print(f"Duplicate rows after cleaning: {df.duplicated().sum()}")

print("\nData cleaning completed!")
print("\nCleaned data shape:", df.shape)

## Step 5: Feature Engineering for Monthly Aggregation

Extract year-month period from timestamp to enable monthly grouping and analysis.

In [ ]:
# Extract year-month period from timestamp for grouping
df['year_month'] = df['timestamp'].dt.to_period('M')

print("Year-month periods in dataset:")
print(df['year_month'].unique())
print(f"\nNumber of unique months: {df['year_month'].nunique()}")
print(f"Number of unique locations: {df['Location'].nunique()}")

## Step 6: Calculate Monthly Averages

Group data by Location and year_month to calculate monthly averages for all environmental parameters.

In [ ]:
# Group data by Location and year_month to calculate monthly averages
monthly_avg = df.groupby(['Location', 'year_month']).agg({
    'Turbidity': 'mean',
    'temperature': 'mean',
    'DO': 'mean',
    'BOD': 'mean',
    'Lead': 'mean',
    'Mercury': 'mean',
    'Arsenic': 'mean',
    'Pollution_Level': 'mean'
}).reset_index()

print("Monthly averages calculated:")
display(monthly_avg.head(10))

print(f"\nTotal monthly records: {len(monthly_avg)}")

## Step 7: Create Visualization 1 - Time Series of Heavy Metal Concentrations

This chart shows trends of toxic heavy metals (Lead, Mercury, Arsenic) over time by location. Critical for environmental monitoring and public health assessments.

In [ ]:
# Create figure for the first chart
fig1, ax1 = plt.subplots(figsize=(14, 6))

# Plot heavy metal concentrations over time for each location
for location in df['Location'].unique():
    location_data = df[df['Location'] == location]
    monthly_location = location_data.groupby('year_month').agg({
        'Lead': 'mean',
        'Mercury': 'mean',
        'Arsenic': 'mean'
    })
    
    plt.plot(monthly_location.index.to_timestamp(), monthly_location['Lead'],
             label=f'{location} - Lead', linewidth=2, marker='o')
    plt.plot(monthly_location.index.to_timestamp(), monthly_location['Mercury'],
             label=f'{location} - Mercury', linestyle='--', linewidth=2, marker='s')
    plt.plot(monthly_location.index.to_timestamp(), monthly_location['Arsenic'],
             label=f'{location} - Arsenic', linestyle=':', linewidth=2, marker='^')

plt.xlabel('Date', fontsize=12)
plt.ylabel('Concentration (mg/L or ppb)', fontsize=12)
plt.title('Monthly Average Heavy Metal Concentrations by Location', fontsize=16, fontweight='bold')
plt.legend(loc='best', fontsize=10)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('chart1_heavy_metals_timeseries.png', dpi=300, bbox_inches='tight')
plt.show()

## Step 8: Create Visualization 2 - Heatmap of Average Pollution Parameters by Location

This heatmap allows quick comparison of pollution levels across monitoring stations, highlighting hotspots that need immediate attention.

In [ ]:
# Create figure for the second chart
fig2, ax2 = plt.subplots(figsize=(12, 6))

# Calculate average pollution parameters by location
heatmap_data = df.groupby('Location')[['Lead', 'Mercury', 'Arsenic', 'BOD', 'Turbidity']].mean()

# Create heatmap
sns.heatmap(heatmap_data.T, annot=True, fmt='.2f', cmap='YlOrRd',
            cbar_kws={'label': 'Concentration (mg/L or ppb)'}))

plt.title('Average Pollution Parameters by Location', fontsize=16, fontweight='bold')
plt.ylabel('Pollutant', fontsize=12)
plt.xlabel('Monitoring Location', fontsize=12)
plt.tight_layout()
plt.savefig('chart2_pollution_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

# Display the data table
print("\nData used for heatmap:")
display(heatmap_data.round(2))

## Step 9: Create Visualization 3 - Correlation Matrix of Environmental Parameters

This chart reveals relationships between different water quality parameters, helping identify which factors tend to increase or decrease together.

In [ ]:
# Create figure for the third chart
fig3, ax3 = plt.subplots(figsize=(10, 8))

# Calculate correlation matrix
correlation_matrix = df[['Turbidity', 'temperature', 'DO', 'BOD', 'Lead', 'Mercury', 'Arsenic', 'Pollution_Level']].corr()

# Create correlation heatmap
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            cbar_kws={'label': 'Correlation Coefficient'},
            square=True, linewidths=0.5)

plt.title('Correlation Matrix of Environmental Parameters', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('chart3_correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

# Print strong correlations (absolute value > 0.5)
print("\nStrong correlations (|r| > 0.5):")
strong_corr = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        corr_val = correlation_matrix.iloc[i, j]
        if abs(corr_val) > 0.5:
            strong_corr.append((correlation_matrix.columns[i], correlation_matrix.columns[j], corr_val))

for var1, var2, corr in strong_corr:
    print(f"{var1} vs {var2}: {corr:.3f}")

## Step 10: Export Cleaned Data

Save the cleaned dataset and monthly averages to new CSV files for further analysis or reporting.

In [ ]:
# Save cleaned dataset and monthly averages to new CSV files
df.to_csv('cleaned_water_quality_data.csv', index=False)
monthly_avg.to_csv('monthly_averages.csv', index=False)

print("Cleaned data saved to 'cleaned_water_quality_data.csv'")
print("Monthly averages saved to 'monthly_averages.csv'")
print("\nAll visualizations saved as PNG files in the current directory.")
print("\nAnalysis completed successfully!")

## Summary of Findings

Add your key observations here after running the analysis:

1. **Heavy Metal Trends**: 
   - Describe any temporal patterns in Lead, Mercury, or Arsenic concentrations
   - Note any locations with consistently high levels

2. **Spatial Variations**:
   - Which location has the highest pollution levels?
   - Are there significant differences between monitoring stations?

3. **Parameter Relationships**:
   - What parameters are strongly correlated?
   - Are there any unexpected relationships?

4. **Recommendations**:
   - Which locations need immediate attention?
   - What additional monitoring might be needed?